# Ham log reports

Starter reports over a CQRLOG/ADIF log converted with **adif2sqlite.py** and
**make_views.py**. Set `DB_PATH` below, then **Run All**.

Tables: `qso` (raw), `qso_deduped` (duplicates removed), `qso_typed`
(deduped + numeric `freq_mhz`/`tx_pwr_w` and `year`/`month`). These reports use
`qso_typed`. Edit freely - this is a starting point for your own SQL and plots.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Path to the database built by adif2sqlite.py + make_views.py
DB_PATH = "hamlog.db"

con = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM qso_typed", con, parse_dates=["qso_ts"])
print(f"{len(df):,} QSOs   {df['qso_ts'].min():%Y-%m-%d} .. {df['qso_ts'].max():%Y-%m-%d}")
df.head()

## QSOs per year

In [ ]:
by_year = df.groupby("year").size()
ax = by_year.plot(kind="bar", figsize=(10, 4), title="QSOs per year")
ax.set_xlabel("year"); ax.set_ylabel("QSOs")
plt.tight_layout(); plt.show()
by_year

## QSOs per month

In [ ]:
by_month = df.set_index("qso_ts").resample("MS").size()
ax = by_month.plot(figsize=(12, 4), title="QSOs per month")
ax.set_xlabel(""); ax.set_ylabel("QSOs")
plt.tight_layout(); plt.show()

## Band × mode

In [ ]:
# bands ordered high-frequency first
band_order = (df.dropna(subset=["freq_mhz"])
                .groupby("band")["freq_mhz"].min()
                .sort_values(ascending=False).index)
mx = (df.pivot_table(index="band", columns="mode", values="id",
                     aggfunc="count", fill_value=0)
        .reindex([b for b in band_order if b in set(df["band"])]))
mx["TOTAL"] = mx.sum(axis=1)
mx

## Modes

In [ ]:
top_modes = df["mode"].value_counts().head(12)
ax = top_modes.sort_values().plot(kind="barh", figsize=(8, 5), title="QSOs by mode")
ax.set_xlabel("QSOs")
plt.tight_layout(); plt.show()
top_modes.to_frame("QSOs")

## Geography (DXCC, continents, states)

In [ ]:
def nonblank(col):
    return df[col].replace("", pd.NA) if col in df.columns else pd.Series(dtype="object")

print(f"DXCC entities worked: {nonblank('dxcc').nunique()}")
print(f"CQ zones worked:      {nonblank('cqz').nunique()}")

display(nonblank("cont").value_counts().rename("QSOs").to_frame())

states = nonblank("state").value_counts().head(15).rename("QSOs").to_frame()
if len(states):
    display(states)

## Most-worked callsigns

In [ ]:
df["call"].value_counts().head(15).rename("QSOs").to_frame()

## Confirmation rates (LoTW / eQSL / card)

In [ ]:
def conf_rate(col):
    if col not in df.columns:
        return None
    return (df[col].fillna("").str.upper() == "Y").mean()

rates = {"LoTW": conf_rate("lotw_qsl_rcvd"),
         "eQSL": conf_rate("eqsl_qsl_rcvd"),
         "Card (QSL)": conf_rate("qsl_rcvd")}
rates = {k: round(v * 100, 1) for k, v in rates.items() if v is not None}
ax = pd.Series(rates).plot(kind="bar", figsize=(6, 4), title="Confirmation rate (%)", ylim=(0, 100))
ax.set_ylabel("% confirmed")
plt.tight_layout(); plt.show()
pd.Series(rates).rename("confirmed %").to_frame()

---
Write your own: `pd.read_sql("SELECT ... FROM qso_typed", con)`. For duplicates
and data-quality checks, see `find_dupes.py`.